# 👩‍💻 **Prompt LLMs for Structured Output + Simulated Function Use**

**Time Estimate:** 45 minutes

## 📋 **Overview**

This activity focuses on enhancing your ability to craft effective prompts for generating structured outputs such as JSON/XML, and simulating logical processes with Large Language Models (LLMs). These skills are vital for developing applications that require precise data manipulation, common in fields such as software development and data analysis.

In this activity, you will create and refine prompts to guide LLMs for predictable outputs, and simulate function calls to ensure practical data logic using free, accessible models.

## 🎯 **Learning Outcomes**

By the end of this lab, you will be able to:

- Design prompts that yield structured JSON/XML outputs.
- Simulate logical operations using LLM-generated outputs.
- Refine prompt engineering skills to ensure accuracy and consistency.

In [1]:
!pip uninstall -y tensorflow tensorflow-cpu tf-keras

Found existing installation: tensorflow 2.14.0
Uninstalling tensorflow-2.14.0:
  Successfully uninstalled tensorflow-2.14.0


In [2]:
!pip install -q "transformers>=4.41,<5.0" sentencepiece



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


## Task 1: Setup and Basic JSON Generation [15 minutes]

In [1]:
import os

# Tell Transformers not to use TensorFlow
os.environ["USE_TF"] = "0"

import torch
from transformers import pipeline
import json
import re

1. Setup a free local LLM and create helper functions for JSON parsing.
2. Test basic structured output generation.

In [2]:
import torch
from transformers import pipeline
import json
import re

generator = pipeline (
"text2text-generation",
model="google/flan-t5-small",
device=-1)

def extract_json(text):
    match = re.search(r'\{.*\}',text,re.DOTALL)
    
    if not match:
        return None
    
    json_text = match.group(0)
    
    try:
        return json.loads(json_text)
    except json.JSONDecodeError:
        return None
    
def generate(prompt,max_new_tokens=200):
        result = generator(
        prompt,
        max_new_tokens = max_new_tokens,
        do_sample=False,
        temperature=0.2
        )
        return result[0]["generated_text"]
    
prompt = """
Generate JSON.

Input:
name = Aayush
age = 20
city = Kochi

Output only:
{"name":"Aayush","age":20,"city":"Kochi"}
"""

output = generate(prompt)

print("Raw model output:")
print(output)

data = extract_json(output)

print("\nParsed JSON:")
print(data)

if data is not None:
    required_fields = ["name", "age", "city"]

    if all(field in data for field in required_fields):
        print("JSON structure is valid.")
    else:
        print("Missing required fields.")
else:
    print("Invalid JSON generated.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw model output:
Aayush is 20 years old and is in Kochi.

Parsed JSON:
None
Invalid JSON generated.


🔍 **Practice**

Test your setup with a simple prompt. Consider:

- How well the model generates structured output.
- The quality of JSON formatting from smaller models.

✅ **Success Checklist**

- Model loads without errors.
- JSON parsing helper function works correctly.
- Basic structured output generation tested.

💡 **Key Points**

- Smaller models require more explicit prompting for structured output.
- JSON parsing tools help handle imperfect model outputs.
- Clear examples in prompts improve consistency.

❗ **Common Mistakes to Avoid**

- Using high temperature for structured tasks (use 0.1-0.3 instead).
- Not providing exact format examples in prompts.
- Forgetting to validate generated JSON structure.

## Task 2: Travel Itinerary JSON Generation [15 minutes]
Create prompts that generate well-structured JSON for travel itineraries.
1. Design a prompt with exact JSON structure examples.
2. Generate travel itineraries for multiple destinations.
3. Validate the consistency of generated JSON.

In [3]:
# ============================================
# TASK 2: TRAVEL ITINERARY JSON GENERATION
# ============================================

def generate_itinerary(destination, days):

    prompt = f"""
Generate a travel itinerary.

Return ONLY valid JSON.
Do not write explanations.
Do not use Markdown.

Use exactly this structure:

{{
  "destination": "string",
  "duration_days": 0,
  "activities": [
    {{
      "day": 1,
      "activity": "string",
      "location": "string"
    }}
  ]
}}

Example:

{{
  "destination": "Paris",
  "duration_days": 3,
  "activities": [
    {{
      "day": 1,
      "activity": "Visit Eiffel Tower",
      "location": "Eiffel Tower"
    }},
    {{
      "day": 2,
      "activity": "Visit Louvre Museum",
      "location": "Louvre Museum"
    }},
    {{
      "day": 3,
      "activity": "Walk along Champs-Elysees",
      "location": "Champs-Elysees"
    }}
  ]
}}

Rules:
1. destination must be "{destination}".
2. duration_days must be {days}.
3. Create one activity for each day.
4. day must start from 1.
5. Every activity must contain day, activity, and location.
6. Use realistic tourist activities.

Destination: {destination}
Duration: {days} days
"""

    output = generate(
        prompt,
        max_new_tokens=400
    )

    return output


# ============================================
# TEST MULTIPLE DESTINATIONS
# ============================================

destinations = [
    ("Paris", 3),
    ("Tokyo", 3),
    ("Dubai", 3)
]


for destination, days in destinations:

    print("=" * 60)
    print("Destination:", destination)

    output = generate_itinerary(
        destination,
        days
    )

    print("\nRaw Output:")
    print(output)

    itinerary = extract_json(output)

    print("\nParsed JSON:")
    print(itinerary)
    
    # ============================================
# VALIDATE ITINERARY
# ============================================

def validate_itinerary(
    data,
    expected_destination,
    expected_days
):

    if data is None:
        return False, "Invalid JSON"

    # Check required top-level fields
    required_fields = [
        "destination",
        "duration_days",
        "activities"
    ]

    for field in required_fields:

        if field not in data:
            return False, f"Missing field: {field}"

    # Check destination
    if data["destination"] != expected_destination:
        return False, "Incorrect destination"

    # Check duration
    if data["duration_days"] != expected_days:
        return False, "Incorrect duration"

    # Check activities
    if not isinstance(data["activities"], list):
        return False, "Activities must be a list"

    # Check each activity
    for activity in data["activities"]:

        required_activity_fields = [
            "day",
            "activity",
            "location"
        ]

        for field in required_activity_fields:

            if field not in activity:
                return False, f"Missing activity field: {field}"

    return True, "Valid itinerary"

# ============================================
# RUN VALIDATION
# ============================================

for destination, days in destinations:

    output = generate_itinerary(
        destination,
        days
    )

    itinerary = extract_json(output)

    valid, message = validate_itinerary(
        itinerary,
        destination,
        days
    )

    print("=" * 60)
    print(destination)
    print("Validation:", message)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Destination: Paris


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Raw Output:
                                                                                                                                                                                                       

Parsed JSON:
None
Destination: Tokyo


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...


Raw Output:
                                                                                                                                                                                                       

Parsed JSON:
None


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Paris
Validation: Invalid JSON


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Tokyo
Validation: Invalid JSON
Dubai
Validation: Invalid JSON


🔍 **Practice**

Test different destinations and analyze the quality of generated content.

✅ **Success Checklist**

- JSON structure matches expected format consistently.
- All required fields are present in generated outputs.
- Content is relevant and realistic for each destination.

💡 **Key Points**

- Providing exact JSON structure examples improves output consistency.
- Lower temperature settings help maintain structure.
- Multiple examples in prompts can improve format adherence.

❗ **Common Mistakes to Avoid**

- Vague structure specifications in prompts.
- Inconsistent field naming across examples.
- Not testing with diverse input variations.

## Task 3: Function Simulation - Order Calculations [15 minutes]
Simulate mathematical function logic using natural language prompts.
1. Create step-by-step calculation prompts.
2. Test order total calculations with tax and shipping.
3. Extract and verify numerical results from LLM outputs.

In [ ]:
# ============================================
# TASK 3: SIMULATE ORDER TOTAL CALCULATION
# ============================================

def calculate_order_total(order):

    subtotal = 0

    for item in order["items"]:
        subtotal += item["price"] * item["quantity"]

    tax = subtotal * order["tax_rate"]

    shipping = order["shipping"]

    total = subtotal + tax + shipping

    return {
        "subtotal": round(subtotal, 2),
        "tax": round(tax, 2),
        "shipping": round(shipping, 2),
        "total": round(total, 2)
    }


# ============================================
# GENERATE ORDER TOTAL USING LLM
# ============================================

def generate_order_total(order):

    prompt = f"""
Calculate the order total.

Return ONLY valid JSON.
Do not write explanations.
Do not use Markdown.

Use exactly this structure:

{{
  "subtotal": 0,
  "tax": 0,
  "shipping": 0,
  "total": 0
}}

Order:
{json.dumps(order)}

Rules:
1. subtotal = sum of price multiplied by quantity.
2. tax = subtotal multiplied by tax_rate.
3. shipping = given shipping cost.
4. total = subtotal + tax + shipping.
5. Return numbers only.
"""

    output = generate(
        prompt,
        max_new_tokens=200
    )

    return output


# ============================================
# TEST ORDER
# ============================================

order = {
    "items": [
        {
            "name": "Laptop",
            "price": 50000,
            "quantity": 1
        },
        {
            "name": "Mouse",
            "price": 1000,
            "quantity": 2
        },
        {
            "name": "Keyboard",
            "price": 2000,
            "quantity": 1
        }
    ],
    "tax_rate": 0.18,
    "shipping": 500
}


# ============================================
# EXPECTED TOTAL USING PYTHON
# ============================================

expected = calculate_order_total(order)

print("=" * 60)
print("EXPECTED CALCULATION")
print("=" * 60)

print("Subtotal:", expected["subtotal"])
print("Tax:", expected["tax"])
print("Shipping:", expected["shipping"])
print("Total:", expected["total"])


# ============================================
# LLM OUTPUT
# ============================================

output = generate_order_total(order)

print("\n" + "=" * 60)
print("LLM RAW OUTPUT")
print("=" * 60)

print(output)


# ============================================
# EXTRACT JSON
# ============================================

result = extract_json(output)

print("\n" + "=" * 60)
print("PARSED JSON")
print("=" * 60)

print(result)


# ============================================
# VERIFY LLM RESULT
# ============================================

if result is None:

    print("\nVerification: Invalid JSON")

else:

    required_fields = [
        "subtotal",
        "tax",
        "shipping",
        "total"
    ]

    valid = True

    for field in required_fields:

        if field not in result:
            print("Missing field:", field)
            valid = False

    if valid:

        print("\nVerification:")

        print(
            "Subtotal correct:",
            result["subtotal"] == expected["subtotal"]
        )

        print(
            "Tax correct:",
            result["tax"] == expected["tax"]
        )

        print(
            "Shipping correct:",
            result["shipping"] == expected["shipping"]
        )

        print(
            "Total correct:",
            result["total"] == expected["total"]
        )

        if (
            result["subtotal"] == expected["subtotal"]
            and result["tax"] == expected["tax"]
            and result["shipping"] == expected["shipping"]
            and result["total"] == expected["total"]
        ):
            print("\nFinal Result: VERIFIED")
        else:
            print("\nFinal Result: INCORRECT")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


EXPECTED CALCULATION
Subtotal: 54000
Tax: 9720.0
Shipping: 500
Total: 64220.0


✅ **Success Checklist**

- Calculation steps are logically sequenced and clear.
- Mathematical operations are performed accurately.
- Final totals can be extracted and verified against actual calculations.

💡 **Key Points**

- Very low temperature (0.1) improves mathematical accuracy.
- Step-by-step examples help guide logical processes.
- Always verify LLM calculations with actual mathematical computation.

❗ **Common Mistakes to Avoid**

- Using high temperature for mathematical tasks.
- Not providing clear calculation examples.
- Trusting LLM math without verification.

🚀 **Next Steps**

In the next module, you will learn how to extend these structured output skills to more complex applications like API integration, database operations, and multi-step workflow automation. This builds on the concepts of prompt engineering and structured data generation to enhance your ability to create robust AI-powered applications.

## 💻 Exemplar Solution

<details>    
<summary><strong>Click HERE to see an exemplar solution</strong></summary>

### Task 1 Solution
    
```python
# Setup free local LLM
model_name = "gpt2"
generator = pipeline('text-generation', 
                    model=model_name, 
                    max_length=300,
                    pad_token_id=50256)

def clean_and_parse_json(text):
    """Helper function to extract and clean JSON from model output"""
    json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if json_match:
        json_str = json_match.group()
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            return {"error": "Could not parse JSON", "raw": json_str}
    return {"error": "No JSON found", "raw": text}

print("✅ Model and helper functions ready")

# Test basic JSON generation
test_prompt = """Create a JSON object with name and age:
{
  "name": "Alice",
  "age": 25
}

Create similar JSON for Bob, age 30:
{"""

response = generator(test_prompt, max_length=50, temperature=0.2, do_sample=True)
generated = response[0]['generated_text'][len(test_prompt):]
result = clean_and_parse_json("{" + generated)
print("Test result:", result)
```

### Task 2 Solution
    
```python
def generate_travel_itinerary(destination, days=3):
    """Generate structured travel itinerary JSON"""
    
    prompt = f"""Create a travel itinerary in JSON format for {destination}. Follow this exact structure:

{{
  "destination": "{destination}",
  "duration_days": {days},
  "departure_date": "2024-06-15",
  "activities": [
    "Visit landmarks",
    "Try local cuisine"
  ],
  "accommodations": "Hotel",
  "budget_estimate": "$500-1000"
}}

Generate similar JSON for {destination}:
{{"""
    
    try:
        response = generator(prompt, 
                           max_length=len(prompt.split()) + 100,
                           temperature=0.3,
                           do_sample=True,
                           pad_token_id=50256)
        
        generated_text = response[0]['generated_text']
        generated_part = generated_text[len(prompt):]
        json_content = "{" + generated_part
        
        return clean_and_parse_json(json_content)
        
    except Exception as e:
        return {"error": str(e), "destination": destination}

# Test with multiple destinations
destinations = ["Paris", "Tokyo", "New York"]

for dest in destinations:
    print(f"\n=== {dest.upper()} ITINERARY ===")
    itinerary = generate_travel_itinerary(dest)
    print(json.dumps(itinerary, indent=2))
```

### Task 3 Solution

```python
def simulate_order_calculation(items_list):
    """Simulate order total calculation using LLM"""
    
    items_text = "\n".join([f"- {item['name']}: ${item['price']} x {item['quantity']}" 
                           for item in items_list])
    
    prompt = f"""Calculate the total cost step by step:

Items:
{items_text}

Example:
- Laptop: $800 x 2 = $1600
- Mouse: $25 x 1 = $25
Subtotal: $1600 + $25 = $1625
Tax (8%): $1625 x 0.08 = $130
Total: $1625 + $130 = $1755

Calculate for items above:
Calculation:"""
    
    try:
        response = generator(prompt,
                           max_length=len(prompt.split()) + 80,
                           temperature=0.1,
                           do_sample=True,
                           pad_token_id=50256)
        
        generated_text = response[0]['generated_text']
        calculation = generated_text[len(prompt):].strip()
        
        # Extract total from calculation
        total_match = re.search(r'Total:\s*\$?([0-9,]+\.?[0-9]*)', calculation, re.IGNORECASE)
        extracted_total = total_match.group(1) if total_match else "Not found"
        
        return {
            "calculation_steps": calculation,
            "extracted_total": extracted_total,
            "items": items_list
        }
        
    except Exception as e:
        return {"error": str(e), "items": items_list}

# Test order calculation
test_order = [
    {"name": "Coffee", "price": 5.99, "quantity": 3},
    {"name": "Sandwich", "price": 8.50, "quantity": 2}
]

print("=== ORDER CALCULATION SIMULATION ===")
result = simulate_order_calculation(test_order)

if "error" not in result:
    print("Items:")
    for item in result["items"]:
        print(f"  {item['name']}: ${item['price']} x {item['quantity']}")
    print(f"\nLLM Calculation:\n{result['calculation_steps']}")
    print(f"\nExtracted Total: ${result['extracted_total']}")
    
    # Verify with actual calculation
    actual_total = sum(item['price'] * item['quantity'] for item in test_order)
    tax = actual_total * 0.08
    final_total = actual_total + tax
    print(f"Actual Total (with 8% tax): ${final_total:.2f}")
else:
    print(f"Error: {result['error']}")

# Enhanced function simulation example
def simulate_route_decision(routes_data):
    """Simulate route optimization decision-making"""
    
    routes_text = "\n".join([
        f"Route {i+1}: {route['name']} - Distance: {route['distance']}km, "
        f"Traffic: {route['traffic']}, Time: {route['time']}min, Cost: ${route['cost']}"
        for i, route in enumerate(routes_data)
    ])
    
    prompt = f"""Choose the optimal route based on distance, traffic, time, and cost.

Available Routes:
{routes_text}

Analysis:
1. Distance comparison:
2. Traffic impact:
3. Time efficiency:
4. Cost consideration:
5. Overall recommendation:

Decision:"""
    
    try:
        response = generator(prompt,
                           max_length=len(prompt.split()) + 100,
                           temperature=0.3,
                           do_sample=True,
                           pad_token_id=50256)
        
        generated_text = response[0]['generated_text']
        decision = generated_text[len(prompt):].strip()
        
        # Extract recommended route
        route_match = re.search(r'Route\s+(\d+)', decision, re.IGNORECASE)
        recommended_route = route_match.group(1) if route_match else "Not specified"
        
        return {
            "analysis": decision,
            "recommended_route": recommended_route,
            "routes": routes_data
        }
        
    except Exception as e:
        return {"error": str(e), "routes": routes_data}

# Test route optimization
sample_routes = [
    {"name": "Highway Route", "distance": 45, "traffic": "Light", "time": 35, "cost": 12.50},
    {"name": "City Route", "distance": 32, "traffic": "Heavy", "time": 55, "cost": 8.75}
]

print("\n=== ROUTE OPTIMIZATION SIMULATION ===")
route_result = simulate_route_decision(sample_routes)

if "error" not in route_result:
    print("Available Routes:")
    for i, route in enumerate(route_result["routes"], 1):
        print(f"  Route {i}: {route['name']} - {route['distance']}km, {route['time']}min, ${route['cost']}")
    
    print(f"\nLLM Analysis:\n{route_result['analysis'][:200]}...")
    print(f"\nRecommended Route: {route_result['recommended_route']}")
else:
    print(f"Error: {route_result['error']}")
```
</details>